# Módulo 4 — Consultas tipo SQL con DuckDB 🦆

**Taller Python for Ford 2026**

**SQL** es el lenguaje universal para consultar datos. Si alguna vez usaste
consultas en Access o pediste un reporte "de la base de datos", eso era SQL.

Con **DuckDB** podemos escribir SQL **de verdad** directamente sobre nuestras
tablas de pandas, sin instalar ninguna base de datos.

> 🌉 **Puente:** todo lo del Módulo 3 (filtrar, agrupar, combinar) se puede
> escribir también en SQL. Aquí verás la equivalencia.

In [ ]:
import pandas as pd
import duckdb

ventas = pd.read_csv("../datos/ventas.csv")
concesionarios = pd.read_csv("../datos/concesionarios.csv")

## 1. Tu primera consulta: `SELECT`

DuckDB puede leer directamente las variables `ventas` y `concesionarios`.
La estructura básica de SQL es:

```sql
SELECT columnas
FROM tabla
```

`*` significa "todas las columnas". `LIMIT 5` = solo las primeras 5 filas.

In [ ]:
duckdb.sql("SELECT * FROM ventas LIMIT 5")

In [ ]:
# Solo algunas columnas:
duckdb.sql("SELECT modelo, monto_venta, metodo_pago FROM ventas LIMIT 5")

## 2. Filtrar con `WHERE` (= filtrar filas)

`WHERE` es el equivalente al Autofiltro / a `tabla[condición]` de pandas.

In [ ]:
duckdb.sql("""
    SELECT modelo, monto_venta
    FROM ventas
    WHERE modelo = 'Mustang'
    LIMIT 10
""")

In [ ]:
# Varias condiciones con AND / OR
duckdb.sql("""
    SELECT modelo, monto_venta, metodo_pago
    FROM ventas
    WHERE monto_venta > 1000000 AND metodo_pago = 'Contado'
    LIMIT 10
""")

## 3. Agrupar con `GROUP BY` (= groupby / tabla dinámica)

`SUM`, `AVG`, `COUNT` son las operaciones. `GROUP BY` dice por qué columna agrupar.

In [ ]:
duckdb.sql("""
    SELECT modelo,
           COUNT(*)        AS num_ventas,
           SUM(monto_venta) AS total,
           AVG(monto_venta) AS promedio
    FROM ventas
    GROUP BY modelo
    ORDER BY total DESC
""")

## 4. Combinar tablas con `JOIN` (= merge / BUSCARV)

Unimos `ventas` con `concesionarios` por `id_concesionario` para poder
analizar por **región**.

In [ ]:
duckdb.sql("""
    SELECT c.region,
           SUM(v.monto_venta) AS total_ventas
    FROM ventas AS v
    JOIN concesionarios AS c
      ON v.id_concesionario = c.id_concesionario
    GROUP BY c.region
    ORDER BY total_ventas DESC
""")

## 5. De SQL a un DataFrame de pandas

Si quieres seguir trabajando el resultado en pandas, agrega `.df()`:

In [ ]:
resultado = duckdb.sql("""
    SELECT region, SUM(monto_venta) AS total
    FROM ventas v JOIN concesionarios c ON v.id_concesionario = c.id_concesionario
    GROUP BY region
""").df()

print(type(resultado))
resultado

## ✅ Resumen del módulo

| Idea | pandas (Módulo 3) | SQL (DuckDB) |
|------|-------------------|--------------|
| Elegir columnas | `tabla[["a","b"]]` | `SELECT a, b` |
| Filtrar filas | `tabla[tabla.x > 0]` | `WHERE x > 0` |
| Agrupar y resumir | `.groupby("x").sum()` | `GROUP BY x` + `SUM()` |
| Combinar tablas | `.merge(otra, on="k")` | `JOIN ... ON` |
| Ordenar | `.sort_values()` | `ORDER BY` |

**Siguiente:** Módulo 5 — convertir estos números en **gráficas**.